# CIFAR-10 / ResNet-18: projected Eq. (1.6) GDA vs SAM vs SGD

This notebook implements a substantially more faithful neural-network comparison than the small FashionMNIST MLP experiment:

- **Dataset:** full CIFAR-10, with a fixed 45k/5k train/validation split and a held-out 10k test set
- **Network:** CIFAR-adapted ResNet-18 (about 11M parameters)
- **Standard vision recipe:** random crop, horizontal flip, normalization, momentum SGD, weight decay, warmup, and cosine decay for SGD/SAM
- **Repeated runs:** three seeds in full mode, identical initialization and data-order seeds across methods
- **Validation-only tuning:** the test set is not used to choose hyperparameters
- **Compute accounting:** training backward passes, diagnostic backward passes, updates, no-gradient forwards, and wall time are tracked separately
- **Diagnostics:** accuracy, error rate, cross-entropy/NLL, calibration error, generalization gap, gradient norm, persistent GDA perturbation norm, and multi-step parameter-space sharpness

The projected Eq. (1.6) GDA implementation intentionally matches the earlier notebook:

$
f_B(x,y)=L_B(x+y),\qquad
x_{t+1}=x_t-\alpha_t\nabla L_B(x_t+y_t),
$

$
y_{t+1}=\Pi_{\lVert y\rVert_2\leq\rho}
\left(y_t+\beta_t\nabla L_B(x_t+y_t)\right).
$

The auxiliary variable \(y\) persists across mini-batches. Every two consecutive mini-batches share one randomly selected Eq. (1.6) branch:

$
\begin{array}{ll}
\text{A (probability }p_{\rm desc}\text{):}&(\alpha,\beta)=(h,-h),(0,h),\\
\text{B (probability }1-p_{\rm desc}\text{):}&(\alpha,\beta)=(-h,h),(h,0).
\end{array}
$

Set `gda_descent_prob` in `DEFAULTS` to any value in `[0, 1]`. It controls the probability of selecting branch A for each complete two-step pair; branch B automatically receives the complementary probability. The default `0.5` recovers the original Eq. (1.6) experiment.

This is **not** SAM: SAM rebuilds a normalized perturbation on every batch and uses two gradients; GDA uses one gradient and a persistent projected ascent variable.

> Start with `RUN_MODE = "smoke"`. A full 200-epoch, three-seed run with tuning is deliberately expensive—especially because SAM needs two backward passes per update.

In [36]:
from pathlib import Path
import copy
import json
import math
import os
import random
import time
from contextlib import contextmanager, nullcontext

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models

plt.rcParams.update({
    "figure.figsize": (7.5, 4.8),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 11,
})

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

ROOT = Path.cwd()
DATA_DIR = ROOT / "data"
RESULTS_DIR = ROOT / "results_cifar10_resnet18_eq16"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("PyTorch:", torch.__version__)
print("device:", DEVICE)
print("results:", RESULTS_DIR)

PyTorch: 2.12.0
device: mps
results: /Users/hooma555/Desktop/Post_Doc/Papers/Singleshot_SAM/results_cifar10_resnet18_eq16


## Configuration and comparison protocols

`canonical` is the primary protocol. It gives SGD and SAM the conventional momentum/weight-decay/cosine recipe used in serious CIFAR experiments, while keeping Eq. (1.6) GDA faithful to the supplied notebook: raw signed updates, no momentum, no weight decay, and constant \(h\).

This is method-faithful but not optimizer-component-isolated. To isolate only the update rules, select `isolated`, which removes momentum, weight decay, and cosine scheduling from SGD/SAM as well. Reporting both protocols is ideal if compute permits.

`gda_descent_prob` controls the Eq. (1.6) branch mixture: branch A (descent/no-op for the model variable) has probability `gda_descent_prob`, while branch B (ascent/descent) has probability `1 - gda_descent_prob`. This changes the mixture of two-step strategies while preserving the internal structure of each branch.

The full run tunes critical hyperparameters on validation data. Pilot tuning is intentionally modest; for publication-quality conclusions, enlarge the grids and tuning budget before running the final seeds.

In [129]:
RUN_MODE = "smoke"       # "smoke" or "full"
PROTOCOL = "isolated"   # "canonical" or "isolated"

COMMON = {
    "batch_size": 1024,
    "val_size": 5_000,
    "split_seed": 2026,
    # Notebook-safe on macOS spawn; avoids non-importable worker callbacks.
    "num_workers": 0,
    "sharpness_rho": 0.05,
    "sharpness_steps": 10,
    "sharpness_batch_size": 256,
    "ece_bins": 15,
    "sam_freeze_bn_second_pass": True,
    "save_checkpoints": False,
}

MODES = {
    "smoke": {
        "epochs": 20,
        "seeds": [0],
        "train_limit": 4096,
        "val_limit": 1024,
        "test_limit": 1024,
        "sharpness_every": 1,
        "run_tuning": False,
        "tune_epochs": 5,
    },
    "full": {
        "epochs": 200,
        "seeds": [0, 1, 2],
        "train_limit": None,
        "val_limit": None,
        "test_limit": None,
        "sharpness_every": 10,
        "run_tuning": True,
        "tune_epochs": 30,
    },
}

CFG = {**COMMON, **MODES[RUN_MODE]}

# Defaults are used directly in smoke mode and are starting points for tuning.
DEFAULTS = {
    "sgd_lr": 0.10,
    "sam_lr": 0.10,
    "sam_rho": 0.05,
    "gda_h": 0.1,
    "gda_rho": 0.01,
    # P(branch A: x descends, then does not update). Branch B gets 1-p.
    "gda_descent_prob": 0.5,
    "momentum": 0.95,
    "weight_decay": 5e-4,
    "warmup_epochs": 5,
}

TUNING_GRIDS = {
    "sgd_lr": [0.05, 0.10, 0.20],
    "sam_rho": [0.025, 0.05, 0.10],
    # Eq. (1.6) has substantial cancellation: its average first-order x drift
    # is roughly h/4 per mini-batch, so the grid extends above ordinary SGD LRs.
    "gda_h": [0.01, 0.05, 0.10, 0.20, 0.40, 0.8],
}

assert RUN_MODE in MODES
assert PROTOCOL in {"canonical", "isolated"}
print(json.dumps({"run_mode": RUN_MODE, "protocol": PROTOCOL, **CFG}, indent=2))

{
  "run_mode": "smoke",
  "protocol": "isolated",
  "batch_size": 1024,
  "val_size": 5000,
  "split_seed": 2026,
  "num_workers": 0,
  "sharpness_rho": 0.05,
  "sharpness_steps": 10,
  "sharpness_batch_size": 256,
  "ece_bins": 15,
  "sam_freeze_bn_second_pass": true,
  "save_checkpoints": false,
  "epochs": 20,
  "seeds": [
    0
  ],
  "train_limit": 4096,
  "val_limit": 1024,
  "test_limit": 1024,
  "sharpness_every": 1,
  "run_tuning": false,
  "tune_epochs": 5
}


## Reproducibility, CIFAR-10 data, and ResNet-18

The validation split is fixed independently of training seeds. Separate dataset objects ensure that validation examples never receive training augmentation. Recreating loaders with the same seed gives every method the same shuffled indices and augmentation RNG stream within a seed.

In [130]:
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2470, 0.2435, 0.2616)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if torch.backends.cudnn.is_available():
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def fixed_split_indices(val_size, split_seed):
    rng = np.random.default_rng(split_seed)
    order = rng.permutation(50_000)
    return order[val_size:].tolist(), order[:val_size].tolist()


def take_limit(indices, limit):
    return indices if limit is None else indices[: min(limit, len(indices))]


def make_loaders(seed, cfg=CFG, download=True):
    train_tf = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    eval_tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])

    augmented = datasets.CIFAR10(DATA_DIR, train=True, download=download, transform=train_tf)
    clean_train = datasets.CIFAR10(DATA_DIR, train=True, download=download, transform=eval_tf)
    test_ds = datasets.CIFAR10(DATA_DIR, train=False, download=download, transform=eval_tf)

    train_idx, val_idx = fixed_split_indices(cfg["val_size"], cfg["split_seed"])
    train_idx = take_limit(train_idx, cfg["train_limit"])
    val_idx = take_limit(val_idx, cfg["val_limit"])
    test_idx = take_limit(list(range(len(test_ds))), cfg["test_limit"])

    gen = torch.Generator().manual_seed(seed)
    loader_kwargs = dict(
        num_workers=cfg["num_workers"],
        pin_memory=(DEVICE.type == "cuda"),
        worker_init_fn=seed_worker,
        persistent_workers=(cfg["num_workers"] > 0),
    )
    train_loader = DataLoader(
        Subset(augmented, train_idx), batch_size=cfg["batch_size"],
        shuffle=True, generator=gen, **loader_kwargs,
    )
    train_eval_loader = DataLoader(
        Subset(clean_train, train_idx), batch_size=512, shuffle=False, **loader_kwargs,
    )
    val_loader = DataLoader(
        Subset(clean_train, val_idx), batch_size=512, shuffle=False, **loader_kwargs,
    )
    test_loader = DataLoader(
        Subset(test_ds, test_idx), batch_size=512, shuffle=False, **loader_kwargs,
    )
    sharp_train_loader = DataLoader(
        Subset(clean_train, train_idx[: cfg["sharpness_batch_size"]]),
        batch_size=cfg["sharpness_batch_size"], shuffle=False, num_workers=0,
    )
    sharp_val_loader = DataLoader(
        Subset(clean_train, val_idx[: cfg["sharpness_batch_size"]]),
        batch_size=cfg["sharpness_batch_size"], shuffle=False, num_workers=0,
    )
    return {
        "train": train_loader,
        "train_eval": train_eval_loader,
        "val": val_loader,
        "test": test_loader,
        "sharp_train": sharp_train_loader,
        "sharp_val": sharp_val_loader,
    }


def make_model():
    model = models.resnet18(weights=None, num_classes=10)
    # CIFAR images are 32x32; the ImageNet stem downsamples too aggressively.
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    return model


set_seed(0)
_model_info = make_model()
print("trainable parameters:", f"{sum(p.numel() for p in _model_info.parameters() if p.requires_grad):,}")
del _model_info

trainable parameters: 11,173,962


## Parameter-space and BatchNorm helpers

Perturbations affect trainable parameters only—not BatchNorm buffers. SAM updates BatchNorm running statistics on its first, unperturbed pass and freezes them on the second pass, avoiding a method-specific double update. GDA has one training-mode pass at \(x+y\), matching its defined objective.

In [131]:
def params(model):
    return [p for p in model.parameters() if p.requires_grad]


@torch.no_grad()
def zeros_like_params(model):
    return [torch.zeros_like(p) for p in params(model)]


@torch.no_grad()
def clone_params(model):
    return [p.detach().clone() for p in params(model)]


@torch.no_grad()
def max_param_diff(model, saved):
    return max((p - q).abs().max().item() for p, q in zip(params(model), saved))


@torch.no_grad()
def add_perturbation_(model, perturbation):
    for p, e in zip(params(model), perturbation):
        p.add_(e)


@torch.no_grad()
def remove_perturbation_(model, perturbation):
    for p, e in zip(params(model), perturbation):
        p.sub_(e)


@torch.no_grad()
def vector_norm(tensors):
    if not tensors:
        return torch.tensor(0.0, device=DEVICE)
    return torch.sqrt(sum(torch.sum(t.detach().float() ** 2) for t in tensors))


@torch.no_grad()
def grad_norm(model):
    return vector_norm([p.grad for p in params(model) if p.grad is not None])


@torch.no_grad()
def normalized_grad(model, eps=1e-12):
    norm = grad_norm(model)
    return [
        torch.zeros_like(p) if p.grad is None else p.grad.detach().clone() / (norm + eps)
        for p in params(model)
    ]


@torch.no_grad()
def project_l2_ball_(perturbation, rho, eps=1e-12):
    norm = vector_norm(perturbation)
    if norm.item() > rho:
        scale = rho / (norm.item() + eps)
        for e in perturbation:
            e.mul_(scale)
    return vector_norm(perturbation).item()


@contextmanager
def freeze_batchnorm_running_stats(model):
    saved = []
    for module in model.modules():
        if isinstance(module, nn.modules.batchnorm._BatchNorm):
            saved.append((module, module.momentum))
            module.momentum = 0.0
    try:
        yield
    finally:
        for module, momentum in saved:
            module.momentum = momentum


def loss_and_logits(model, x, y):
    logits = model(x)
    return F.cross_entropy(logits, y), logits

## Optimizer steps

SGD and SAM use PyTorch's momentum-SGD state. SAM performs the conventional normalized first-gradient perturbation and updates the unperturbed parameters using the second gradient.

Eq. (1.6) GDA deliberately does **not** use a base optimizer: the signed update is applied directly, exactly as in the supplied notebook. Negative \(\alpha\) therefore really means ascent, rather than an unusual input to a momentum buffer.

In [132]:
def cosine_warmup_lambda(step, total_steps, warmup_steps):
    if warmup_steps > 0 and step < warmup_steps:
        return max((step + 1) / warmup_steps, 1e-8)
    progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
    progress = min(max(progress, 0.0), 1.0)
    return 0.5 * (1.0 + math.cos(math.pi * progress))


def build_base_optimizer(model, spec, steps_per_epoch, epochs):
    optimizer = torch.optim.SGD(
        model.parameters(), lr=spec["lr"], momentum=spec["momentum"],
        weight_decay=spec["weight_decay"], nesterov=(spec["momentum"] > 0),
    )
    scheduler = None
    if spec.get("cosine", False):
        total_steps = steps_per_epoch * epochs
        warmup_steps = steps_per_epoch * spec.get("warmup_epochs", 0)
        scheduler = torch.optim.lr_scheduler.LambdaLR(
            optimizer,
            lr_lambda=lambda s: cosine_warmup_lambda(s, total_steps, warmup_steps),
        )
    return optimizer, scheduler


def step_sgd(model, x, y, state, spec):
    optimizer, scheduler = state["optimizer"], state["scheduler"]
    optimizer.zero_grad(set_to_none=True)
    loss, _ = loss_and_logits(model, x, y)
    loss.backward()
    gnorm = grad_norm(model).item()
    optimizer.step()
    if scheduler is not None:
        scheduler.step()
    return {"objective_loss": loss.item(), "grad_norm": gnorm, "grad_evals": 1,
            "alpha": np.nan, "beta": np.nan, "eps_norm": 0.0}


def step_sam(model, x, y, state, spec):
    optimizer, scheduler = state["optimizer"], state["scheduler"]

    # First gradient at unperturbed parameters.
    optimizer.zero_grad(set_to_none=True)
    base_loss, _ = loss_and_logits(model, x, y)
    base_loss.backward()
    direction = normalized_grad(model)
    perturbation = [spec["rho"] * d for d in direction]

    saved = clone_params(model)
    add_perturbation_(model, perturbation)
    optimizer.zero_grad(set_to_none=True)
    bn_context = freeze_batchnorm_running_stats(model) if spec["freeze_bn_second_pass"] else nullcontext()
    with bn_context:
        perturbed_loss, _ = loss_and_logits(model, x, y)
        perturbed_loss.backward()
    remove_perturbation_(model, perturbation)
    if max_param_diff(model, saved) > 2e-6:
        raise RuntimeError("SAM failed to restore the unperturbed parameters")

    gnorm = grad_norm(model).item()
    optimizer.step()
    if scheduler is not None:
        scheduler.step()
    return {"objective_loss": perturbed_loss.item(), "grad_norm": gnorm, "grad_evals": 2,
            "alpha": np.nan, "beta": np.nan,
            "eps_norm": vector_norm(perturbation).item()}


def sample_next_eq16_step(state, h, descent_branch_prob=0.5):
    if not 0.0 <= descent_branch_prob <= 1.0:
        raise ValueError("descent_branch_prob must lie in [0, 1]")
    if state.get("pair_pos", 2) >= 2:
        branch = 0 if state["rng"].random() < descent_branch_prob else 1
        pair = [(h, -h), (0.0, h)] if branch == 0 else [(-h, h), (h, 0.0)]
        state["pair"] = pair
        state["pair_pos"] = 0
        state.setdefault("branches", []).append(branch)
        state.setdefault("pair_h", []).append(float(h))
    alpha, beta = state["pair"][state["pair_pos"]]
    state["pair_pos"] += 1
    state.setdefault("alpha_history", []).append(float(alpha))
    state.setdefault("beta_history", []).append(float(beta))
    return float(alpha), float(beta)


@torch.no_grad()
def signed_x_update_(model, alpha):
    if alpha == 0.0:
        return
    for p in params(model):
        if p.grad is not None:
            p.add_(p.grad, alpha=-alpha)


@torch.no_grad()
def signed_y_update_(perturbation, model, beta):
    if beta == 0.0:
        return
    for e, p in zip(perturbation, params(model)):
        if p.grad is not None:
            e.add_(p.grad, alpha=beta)


def step_eq16_gda(model, x, y, state, spec):
    if "eps" not in state:
        state["eps"] = zeros_like_params(model)
    alpha, beta = sample_next_eq16_step(
        state, spec["h"], spec.get("descent_branch_prob", 0.5)
    )
    eps = state["eps"]

    saved = clone_params(model)
    add_perturbation_(model, eps)
    model.zero_grad(set_to_none=True)
    loss, _ = loss_and_logits(model, x, y)
    loss.backward()
    gnorm = grad_norm(model).item()
    remove_perturbation_(model, eps)
    if max_param_diff(model, saved) > 2e-6:
        raise RuntimeError("GDA failed to restore x before its signed update")

    # Same gradient, evaluated at x+y, updates both variables.
    signed_y_update_(eps, model, beta)
    eps_norm = project_l2_ball_(eps, spec["rho"])
    signed_x_update_(model, alpha)
    if eps_norm > spec["rho"] + 1e-6:
        raise RuntimeError("GDA projection invariant failed")
    return {"objective_loss": loss.item(), "grad_norm": gnorm, "grad_evals": 1,
            "alpha": alpha, "beta": beta, "eps_norm": eps_norm}


STEP_FUNCTIONS = {"sgd": step_sgd, "sam": step_sam, "eq16_gda": step_eq16_gda}

## Evaluation and sharpness

The reported sharpness is a fixed-batch, multi-step projected-gradient proxy for

$
\max_{\lVert e\rVert_2\leq\rho_{\rm eval}} L(w+e)-L(w).
$

It is evaluated with BatchNorm in inference mode and never affects training. This quantity is parameterization-dependent, so it should be interpreted alongside—not instead of—test performance. Diagnostic backward passes are counted separately from training gradients.

In [133]:
@torch.no_grad()
def expected_calibration_error(confidences, correctness, n_bins=15):
    edges = torch.linspace(0, 1, n_bins + 1, device=confidences.device)
    ece = torch.zeros((), device=confidences.device)
    for lo, hi in zip(edges[:-1], edges[1:]):
        mask = (confidences > lo) & (confidences <= hi)
        if mask.any():
            ece += mask.float().mean() * (correctness[mask].float().mean() - confidences[mask].mean()).abs()
    return ece.item()


@torch.no_grad()
def evaluate(model, loader, ece_bins=15):
    was_training = model.training
    model.eval()
    total_loss, total_correct, total = 0.0, 0, 0
    all_conf, all_correct = [], []
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = model(x)
        total_loss += F.cross_entropy(logits, y, reduction="sum").item()
        probs = logits.softmax(dim=1)
        conf, pred = probs.max(dim=1)
        correct = pred.eq(y)
        total_correct += correct.sum().item()
        total += y.numel()
        all_conf.append(conf)
        all_correct.append(correct)
    confidence = torch.cat(all_conf)
    correctness = torch.cat(all_correct)
    result = {
        "nll": total_loss / total,
        "accuracy": total_correct / total,
        "error": 1.0 - total_correct / total,
        "ece": expected_calibration_error(confidence, correctness, ece_bins),
        "forward_batches": len(loader),
    }
    model.train(was_training)
    return result


@torch.no_grad()
def post_update_batch_metrics(model, x, y):
    was_training = model.training
    model.eval()
    logits = model(x)
    loss = F.cross_entropy(logits, y, reduction="sum").item()
    correct = logits.argmax(dim=1).eq(y).sum().item()
    model.train(was_training)
    return loss, correct


def sharpness_proxy(model, x, y, rho=0.05, steps=10, eps=1e-12):
    was_training = model.training
    model.eval()
    perturbation = zeros_like_params(model)
    original = clone_params(model)

    with torch.no_grad():
        base_loss = F.cross_entropy(model(x), y).item()

    # A slightly aggressive PGD step reaches and explores the boundary.
    step_size = 1.5 * rho / max(steps, 1)
    for _ in range(steps):
        add_perturbation_(model, perturbation)
        model.zero_grad(set_to_none=True)
        loss = F.cross_entropy(model(x), y)
        loss.backward()
        direction = normalized_grad(model, eps=eps)
        remove_perturbation_(model, perturbation)
        with torch.no_grad():
            for e, d in zip(perturbation, direction):
                e.add_(d, alpha=step_size)
            project_l2_ball_(perturbation, rho, eps=eps)

    add_perturbation_(model, perturbation)
    with torch.no_grad():
        perturbed_loss = F.cross_entropy(model(x), y).item()
    remove_perturbation_(model, perturbation)
    model.zero_grad(set_to_none=True)
    if max_param_diff(model, original) > 2e-6:
        raise RuntimeError("Sharpness diagnostic failed to restore parameters")
    model.train(was_training)

    absolute = perturbed_loss - base_loss
    return {
        "base_loss": base_loss,
        "perturbed_loss": perturbed_loss,
        "sharpness": absolute,
        "relative_sharpness": absolute / (abs(base_loss) + eps),
        "grad_evals": steps,
    }


def fixed_batch(loader):
    x, y = next(iter(loader))
    return x.to(DEVICE), y.to(DEVICE)

## Method specifications and invariants

The schedule test verifies exact branch consumption and empirically checks the 50/50 branch sampler. One-step tests use a tiny network so they remain cheap.

In [134]:
def make_specs(values=None, protocol=PROTOCOL):
    v = {**DEFAULTS, **(values or {})}
    canonical = protocol == "canonical"
    momentum = v["momentum"] if canonical else 0.0
    weight_decay = v["weight_decay"] if canonical else 0.0
    cosine = canonical
    warmup = v["warmup_epochs"] if canonical else 0
    return {
        "SGD": {
            "key": "sgd", "lr": v["sgd_lr"], "momentum": momentum,
            "weight_decay": weight_decay, "cosine": cosine, "warmup_epochs": warmup,
        },
        "SAM": {
            "key": "sam", "lr": v["sam_lr"], "rho": v["sam_rho"],
            "momentum": momentum, "weight_decay": weight_decay,
            "cosine": cosine, "warmup_epochs": warmup,
            "freeze_bn_second_pass": CFG["sam_freeze_bn_second_pass"],
        },
        f"Proj-Eq1.6-GDA(h={v['gda_h']:g},rho={v['gda_rho']:g},"
        f"p_desc={v['gda_descent_prob']:g})": {
            "key": "eq16_gda", "h": v["gda_h"], "rho": v["gda_rho"],
            "descent_branch_prob": v["gda_descent_prob"],
            "cosine": False, "momentum": 0.0, "weight_decay": 0.0,
        },
    }


def verify_eq16_history(state, atol=1e-12):
    alpha = state.get("alpha_history", [])
    beta = state.get("beta_history", [])
    branches = state.get("branches", [])
    pair_h = state.get("pair_h", [])
    for j in range(len(alpha) // 2):
        h, branch = pair_h[j], branches[j]
        got = (alpha[2*j], beta[2*j], alpha[2*j+1], beta[2*j+1])
        target = (h, -h, 0.0, h) if branch == 0 else (-h, h, h, 0.0)
        if not np.allclose(got, target, atol=atol):
            raise AssertionError((j, branch, got, target))
    return True


def schedule_sanity_check():
    # Check endpoints and several mixtures, including the original p=0.5 case.
    for target_probability in [0.0, 0.2, 0.5, 0.8, 1.0]:
        state = {"rng": np.random.default_rng(123)}
        for _ in range(20_000):
            sample_next_eq16_step(
                state, h=0.1, descent_branch_prob=target_probability
            )
        verify_eq16_history(state)
        sampled_probability = 1.0 - np.mean(state["branches"])
        tolerance = 0.0 if target_probability in {0.0, 1.0} else 0.03
        assert abs(sampled_probability - target_probability) <= tolerance, (
            target_probability, sampled_probability
        )
        print(
            f"Eq. (1.6) schedule OK; target P(branch A)={target_probability:.2f}, "
            f"sampled={sampled_probability:.4f}"
        )


class TinyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Flatten(), nn.Linear(3 * 8 * 8, 16), nn.ReLU(), nn.Linear(16, 10))
    def forward(self, x):
        return self.net(x)


def one_step_sanity_checks():
    x = torch.randn(8, 3, 8, 8, device=DEVICE)
    y = torch.randint(0, 10, (8,), device=DEVICE)
    for name, spec in make_specs().items():
        set_seed(7)
        model = TinyNet().to(DEVICE).train()
        state = {"rng": np.random.default_rng(7)}
        if spec["key"] in {"sgd", "sam"}:
            state["optimizer"], state["scheduler"] = build_base_optimizer(model, spec, 1, 1)
        out = STEP_FUNCTIONS[spec["key"]](model, x, y, state, spec)
        expected = 2 if spec["key"] == "sam" else 1
        assert out["grad_evals"] == expected
        if spec["key"] == "eq16_gda":
            assert out["eps_norm"] <= spec["rho"] + 1e-6
        print(f"{name:38s} gradients={out['grad_evals']} eps_norm={out['eps_norm']:.5f}")


schedule_sanity_check()
one_step_sanity_checks()

Eq. (1.6) schedule OK; target P(branch A)=0.00, sampled=0.0000
Eq. (1.6) schedule OK; target P(branch A)=0.20, sampled=0.2030
Eq. (1.6) schedule OK; target P(branch A)=0.50, sampled=0.5049
Eq. (1.6) schedule OK; target P(branch A)=0.80, sampled=0.8127
Eq. (1.6) schedule OK; target P(branch A)=1.00, sampled=1.0000
SGD                                    gradients=1 eps_norm=0.00000
SAM                                    gradients=2 eps_norm=0.05000
Proj-Eq1.6-GDA(h=0.1,rho=0.01,p_desc=0.5) gradients=1 eps_norm=0.01000


## Training engine

All reported evaluation metrics use the unperturbed model \(x\). Training accuracy/loss are measured after each update in inference mode, so the extra logging forward does not alter BatchNorm statistics. This forward is tracked but is not counted as a gradient evaluation.

For model selection, the primary score is validation accuracy. The test set is optional during pilot tuning and is never used to select settings.

In [135]:
def initial_state_for_seed(seed):
    set_seed(seed)
    return copy.deepcopy(make_model().state_dict())


def build_training_state(model, spec, loader_len, epochs, schedule_seed):
    state = {"rng": np.random.default_rng(schedule_seed)}
    if spec["key"] in {"sgd", "sam"}:
        state["optimizer"], state["scheduler"] = build_base_optimizer(model, spec, loader_len, epochs)
    return state


def train_one(seed, method_name, spec, epochs, initial_state_dict,
              record_sharpness=True, evaluate_test=True, verbose=True):
    set_seed(seed)
    loaders = make_loaders(seed)
    model = make_model().to(DEVICE)
    model.load_state_dict(copy.deepcopy(initial_state_dict))
    state = build_training_state(model, spec, len(loaders["train"]), epochs, 100_000 + seed)
    step_fn = STEP_FUNCTIONS[spec["key"]]
    sharp_train_batch = fixed_batch(loaders["sharp_train"])
    sharp_val_batch = fixed_batch(loaders["sharp_val"])

    rows = []
    train_grad_evals = 0
    diagnostic_grad_evals = 0
    updates = 0
    no_grad_train_forwards = 0
    eval_forward_batches = 0
    start = time.perf_counter()

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss, running_correct, running_total = 0.0, 0, 0
        grad_norms, eps_norms, alphas, betas = [], [], [], []

        for x, y in loaders["train"]:
            x, y = x.to(DEVICE), y.to(DEVICE)
            out = step_fn(model, x, y, state, spec)
            train_grad_evals += out["grad_evals"]
            updates += 1

            batch_loss, batch_correct = post_update_batch_metrics(model, x, y)
            no_grad_train_forwards += 1
            running_loss += batch_loss
            running_correct += batch_correct
            running_total += y.numel()
            grad_norms.append(out["grad_norm"])
            eps_norms.append(out["eps_norm"])
            if not np.isnan(out["alpha"]): alphas.append(out["alpha"])
            if not np.isnan(out["beta"]): betas.append(out["beta"])

            if not math.isfinite(out["objective_loss"]):
                raise FloatingPointError(f"Non-finite loss in {method_name}, epoch {epoch}")

        do_sharpness = record_sharpness and (
            epoch == 1 or epoch == epochs or epoch % CFG["sharpness_every"] == 0
        )
        # A clean, non-augmented training-set pass makes the generalization gap
        # interpretable. It is limited to diagnostic epochs to control overhead.
        if do_sharpness:
            clean_train = evaluate(model, loaders["train_eval"], CFG["ece_bins"])
            eval_forward_batches += clean_train["forward_batches"]
        else:
            clean_train = {"nll": np.nan, "accuracy": np.nan, "error": np.nan,
                           "ece": np.nan, "forward_batches": 0}

        val = evaluate(model, loaders["val"], CFG["ece_bins"])
        eval_forward_batches += val["forward_batches"]
        if evaluate_test:
            test = evaluate(model, loaders["test"], CFG["ece_bins"])
            eval_forward_batches += test["forward_batches"]
        else:
            test = {"nll": np.nan, "accuracy": np.nan, "error": np.nan,
                    "ece": np.nan, "forward_batches": 0}

        if do_sharpness:
            st = sharpness_proxy(model, *sharp_train_batch, CFG["sharpness_rho"], CFG["sharpness_steps"])
            sv = sharpness_proxy(model, *sharp_val_batch, CFG["sharpness_rho"], CFG["sharpness_steps"])
            diagnostic_grad_evals += st["grad_evals"] + sv["grad_evals"]
        else:
            st = sv = {"sharpness": np.nan, "relative_sharpness": np.nan, "grad_evals": 0}

        lr = (state["optimizer"].param_groups[0]["lr"]
              if "optimizer" in state else np.nan)
        elapsed = time.perf_counter() - start
        row = {
            "seed": seed, "method": method_name, "protocol": PROTOCOL, "epoch": epoch,
            "train_loss": running_loss / running_total,
            "train_accuracy": running_correct / running_total,
            "train_error": 1.0 - running_correct / running_total,
            "clean_train_nll": clean_train["nll"],
            "clean_train_accuracy": clean_train["accuracy"],
            "clean_train_error": clean_train["error"],
            "val_nll": val["nll"], "val_accuracy": val["accuracy"],
            "val_error": val["error"], "val_ece": val["ece"],
            "test_nll": test["nll"], "test_accuracy": test["accuracy"],
            "test_error": test["error"], "test_ece": test["ece"],
            "generalization_gap": val["error"] - clean_train["error"],
            "train_sharpness": st["sharpness"], "val_sharpness": sv["sharpness"],
            "train_relative_sharpness": st["relative_sharpness"],
            "val_relative_sharpness": sv["relative_sharpness"],
            "mean_grad_norm": float(np.mean(grad_norms)),
            "mean_eps_norm": float(np.mean(eps_norms)),
            "mean_alpha": float(np.mean(alphas)) if alphas else np.nan,
            "mean_beta": float(np.mean(betas)) if betas else np.nan,
            "learning_rate": lr, "gda_h": spec.get("h", np.nan),
            "gda_descent_branch_prob": spec.get("descent_branch_prob", np.nan),
            "gda_ascent_branch_prob": (
                1.0 - spec["descent_branch_prob"]
                if "descent_branch_prob" in spec else np.nan
            ),
            "updates": updates, "train_grad_evals": train_grad_evals,
            "diagnostic_grad_evals": diagnostic_grad_evals,
            "total_grad_evals": train_grad_evals + diagnostic_grad_evals,
            "no_grad_train_forwards": no_grad_train_forwards,
            "eval_forward_batches": eval_forward_batches,
            "wall_time_sec": elapsed,
        }
        rows.append(row)
        if verbose:
            test_text = f" test={100*test['accuracy']:.2f}%" if evaluate_test else ""
            print(f"seed={seed} {method_name:38s} epoch={epoch:03d} "
                  f"val={100*val['accuracy']:.2f}%{test_text} "
                  f"train_grads={train_grad_evals:,} time={elapsed/60:.1f}m")

    if spec["key"] == "eq16_gda":
        verify_eq16_history(state)
    return pd.DataFrame(rows), model

## Optional validation-only pilot tuning

This stage never evaluates the test set. It chooses the setting with the highest mean validation accuracy over the last five pilot epochs. The SAM learning rate follows the selected SGD learning rate; then SAM's \(\rho\) and GDA's \(h\) are tuned separately.

Pilot tuning is a pragmatic compromise, not a claim of exhaustive optimization. Save and report the full tuning table.

In [136]:
def pilot_score(history):
    return history.tail(min(5, len(history)))["val_accuracy"].mean()


def run_pilot(label, candidates, values, tune_seed=20_026):
    init = initial_state_for_seed(tune_seed)
    records = []
    for candidate in candidates:
        trial_values = {**values, label: candidate}
        specs = make_specs(trial_values)
        if label == "sgd_lr":
            method_name = "SGD"
        elif label == "sam_rho":
            method_name = "SAM"
        elif label == "gda_h":
            method_name = next(k for k, v in specs.items() if v["key"] == "eq16_gda")
        spec = specs[method_name]
        print(f"\nPILOT {label}={candidate:g}: {method_name}")
        hist, _ = train_one(
            tune_seed, method_name, spec, CFG["tune_epochs"], init,
            record_sharpness=False, evaluate_test=False, verbose=False,
        )
        score = pilot_score(hist)
        records.append({"parameter": label, "candidate": candidate, "score": score})
        print(f"mean last-five validation accuracy={100*score:.3f}%")
    table = pd.DataFrame(records).sort_values("score", ascending=False)
    selected = float(table.iloc[0]["candidate"])
    return selected, table


selected_values = dict(DEFAULTS)
tuning_tables = []

if CFG["run_tuning"]:
    selected_values["sgd_lr"], table = run_pilot(
        "sgd_lr", TUNING_GRIDS["sgd_lr"], selected_values
    )
    tuning_tables.append(table)
    selected_values["sam_lr"] = selected_values["sgd_lr"]

    selected_values["sam_rho"], table = run_pilot(
        "sam_rho", TUNING_GRIDS["sam_rho"], selected_values
    )
    tuning_tables.append(table)

    selected_values["gda_h"], table = run_pilot(
        "gda_h", TUNING_GRIDS["gda_h"], selected_values
    )
    tuning_tables.append(table)
    tuning_df = pd.concat(tuning_tables, ignore_index=True)
    tuning_df.to_csv(RESULTS_DIR / "validation_tuning.csv", index=False)
else:
    tuning_df = pd.DataFrame()

print("selected values:", json.dumps(selected_values, indent=2))

selected values: {
  "sgd_lr": 0.1,
  "sam_lr": 0.1,
  "sam_rho": 0.05,
  "gda_h": 0.1,
  "gda_rho": 0.01,
  "gda_descent_prob": 0.5,
  "momentum": 0.95,
  "weight_decay": 0.0005,
  "warmup_epochs": 5
}


## Main repeated experiment

Each method begins from the exact same initialization within each seed. Loader and augmentation seeds are also reset per method. Final checkpoints can optionally be saved by setting `save_checkpoints=True`.

In [137]:
METHODS = make_specs(selected_values)
all_histories = []

for seed in CFG["seeds"]:
    init = initial_state_for_seed(seed)
    for method_name, spec in METHODS.items():
        print("\n" + "=" * 100)
        print("MAIN RUN", "seed=", seed, "method=", method_name, "spec=", spec)
        hist, trained_model = train_one(
            seed, method_name, spec, CFG["epochs"], init,
            record_sharpness=True, evaluate_test=True, verbose=True,
        )
        all_histories.append(hist)
        if CFG["save_checkpoints"]:
            safe_name = "".join(c if c.isalnum() or c in "-_" else "_" for c in method_name)
            torch.save(trained_model.state_dict(), RESULTS_DIR / f"checkpoint_seed{seed}_{safe_name}.pt")
        del trained_model
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

history_df = pd.concat(all_histories, ignore_index=True)
history_df.to_csv(RESULTS_DIR / "history.csv", index=False)
with open(RESULTS_DIR / "selected_configuration.json", "w") as f:
    json.dump({"run_mode": RUN_MODE, "protocol": PROTOCOL, "cfg": CFG,
               "selected_values": selected_values, "methods": METHODS}, f, indent=2)

print("saved", RESULTS_DIR / "history.csv")
history_df.tail()


MAIN RUN seed= 0 method= SGD spec= {'key': 'sgd', 'lr': 0.1, 'momentum': 0.0, 'weight_decay': 0.0, 'cosine': False, 'warmup_epochs': 0}
seed=0 SGD                                    epoch=001 val=11.13% test=11.33% train_grads=4 time=0.1m
seed=0 SGD                                    epoch=002 val=15.14% test=14.94% train_grads=8 time=0.3m
seed=0 SGD                                    epoch=003 val=17.29% test=17.29% train_grads=12 time=0.4m
seed=0 SGD                                    epoch=004 val=15.04% test=15.04% train_grads=16 time=0.5m
seed=0 SGD                                    epoch=005 val=24.41% test=24.02% train_grads=20 time=0.7m
seed=0 SGD                                    epoch=006 val=22.46% test=25.39% train_grads=24 time=0.8m
seed=0 SGD                                    epoch=007 val=23.63% test=25.29% train_grads=28 time=1.0m
seed=0 SGD                                    epoch=008 val=24.12% test=27.15% train_grads=32 time=1.1m
seed=0 SGD                       

,seed,method,protocol,epoch,train_loss,train_accuracy,train_error,clean_train_nll,clean_train_accuracy,clean_train_error,...,gda_h,gda_descent_branch_prob,gda_ascent_branch_prob,updates,train_grad_evals,diagnostic_grad_evals,total_grad_evals,no_grad_train_forwards,eval_forward_batches,wall_time_sec
55,0,"Proj-Eq1.6-GDA(h=0.1,rho=0.01,p_desc=0.5)",isolated,16,2.252590,0.172607,0.827393,2.240297,0.157959,0.842041,...,0.1,0.5,0.5,64,64,320,384,64,192,130.582440
56,0,"Proj-Eq1.6-GDA(h=0.1,rho=0.01,p_desc=0.5)",isolated,17,2.230801,0.182373,0.817627,2.231600,0.154541,0.845459,...,0.1,0.5,0.5,68,68,340,408,68,204,138.756246
57,0,"Proj-Eq1.6-GDA(h=0.1,rho=0.01,p_desc=0.5)",isolated,18,2.226809,0.186279,0.813721,2.230921,0.179688,0.820312,...,0.1,0.5,0.5,72,72,360,432,72,216,146.981524
58,0,"Proj-Eq1.6-GDA(h=0.1,rho=0.01,p_desc=0.5)",isolated,19,2.246872,0.183838,0.816162,2.235559,0.181152,0.818848,...,0.1,0.5,0.5,76,76,380,456,76,228,155.078612
59,0,"Proj-Eq1.6-GDA(h=0.1,rho=0.01,p_desc=0.5)",isolated,20,2.222627,0.202393,0.797607,2.224812,0.196533,0.803467,...,0.1,0.5,0.5,80,80,400,480,80,240,163.316591


## Summaries selected without test leakage

For each seed/method, `test_at_best_val` is taken at the epoch with the highest validation accuracy. The test trajectory does not choose the epoch. Final metrics are reported separately.

In [56]:
def per_seed_summary(history):
    rows = []
    for (seed, method), group in history.groupby(["seed", "method"], sort=False):
        group = group.sort_values("epoch")
        chosen = group.loc[group["val_accuracy"].idxmax()]
        final = group.iloc[-1]
        rows.append({
            "seed": seed, "method": method,
            "best_val_epoch": int(chosen["epoch"]),
            "best_val_accuracy": chosen["val_accuracy"],
            "test_accuracy_at_best_val": chosen["test_accuracy"],
            "test_error_at_best_val": chosen["test_error"],
            "test_nll_at_best_val": chosen["test_nll"],
            "test_ece_at_best_val": chosen["test_ece"],
            "final_test_accuracy": final["test_accuracy"],
            "final_test_error": final["test_error"],
            "final_test_nll": final["test_nll"],
            "final_test_ece": final["test_ece"],
            "final_train_sharpness": final["train_sharpness"],
            "final_val_sharpness": final["val_sharpness"],
            "final_generalization_gap": final["generalization_gap"],
            "final_mean_grad_norm": final["mean_grad_norm"],
            "final_mean_eps_norm": final["mean_eps_norm"],
            "updates": int(final["updates"]),
            "train_grad_evals": int(final["train_grad_evals"]),
            "diagnostic_grad_evals": int(final["diagnostic_grad_evals"]),
            "wall_time_sec": final["wall_time_sec"],
        })
    return pd.DataFrame(rows)


per_seed_df = per_seed_summary(history_df)
numeric_cols = [c for c in per_seed_df.columns if c not in {"seed", "method"}]
aggregate_df = per_seed_df.groupby("method")[numeric_cols].agg(["mean", "std"])

per_seed_df.to_csv(RESULTS_DIR / "per_seed_summary.csv", index=False)
aggregate_df.to_csv(RESULTS_DIR / "aggregate_summary.csv")

display_cols = [
    "method", "test_accuracy_at_best_val", "test_error_at_best_val",
    "final_train_sharpness", "final_val_sharpness",
    "train_grad_evals", "diagnostic_grad_evals", "wall_time_sec",
]
display(per_seed_df[display_cols].sort_values(["seed", "test_error_at_best_val"]))
display(aggregate_df)

KeyError: 'seed'

## Paired differences across matched seeds

Paired differences are more informative than comparing two independent means because initialization and data-order seeds are matched. With only three seeds, treat uncertainty estimates as descriptive rather than definitive.

In [ ]:
def paired_differences(summary, metric, baseline="SGD"):
    wide = summary.pivot(index="seed", columns="method", values=metric)
    rows = []
    for method in wide.columns:
        if method == baseline:
            continue
        diff = wide[method] - wide[baseline]
        rows.append({
            "metric": metric, "method_minus_baseline": f"{method} - {baseline}",
            "mean_difference": diff.mean(), "std_difference": diff.std(ddof=1),
            "n_seeds": diff.notna().sum(),
        })
    return pd.DataFrame(rows)


paired_df = pd.concat([
    paired_differences(per_seed_df, "test_accuracy_at_best_val"),
    paired_differences(per_seed_df, "test_error_at_best_val"),
    paired_differences(per_seed_df, "final_train_sharpness"),
], ignore_index=True)
paired_df.to_csv(RESULTS_DIR / "paired_differences.csv", index=False)
paired_df

## Learning curves: epoch-matched and gradient-matched

SAM uses two training gradients per update; SGD and Eq. (1.6) GDA use one. Therefore both panels matter. The gradient-evaluation panel exposes SAM's extra backward-pass cost. Sharpness diagnostics are excluded from the x-axis because they are evaluation overhead, not training signal.

In [ ]:
def plot_metric(df, metric, ylabel, percent=False, sharpness_metric=False):
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
    plot_df = df.dropna(subset=[metric])
    for method in METHODS:
        sub = plot_df[plot_df["method"] == method]
        for ax, xcol, xlabel in [
            (axes[0], "epoch", "Epoch"),
            (axes[1], "train_grad_evals", "Training gradient evaluations"),
        ]:
            grouped = sub.groupby(xcol)[metric].agg(["mean", "std"]).reset_index()
            scale = 100 if percent else 1
            y = scale * grouped["mean"]
            ax.plot(grouped[xcol], y, label=method)
            if grouped["std"].notna().any():
                std = scale * grouped["std"].fillna(0)
                ax.fill_between(grouped[xcol], y - std, y + std, alpha=0.13)
            ax.set_xlabel(xlabel)
            ax.set_ylabel(ylabel)
    axes[0].set_title(f"{ylabel} vs epoch")
    axes[1].set_title(f"{ylabel} vs training gradients")
    axes[1].legend(bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()


plot_metric(history_df, "test_accuracy", "Test accuracy (%)", percent=True)
plot_metric(history_df, "test_error", "Test error (%)", percent=True)
plot_metric(history_df, "test_nll", "Test NLL")
plot_metric(history_df, "train_sharpness", "Training sharpness proxy")
plot_metric(history_df, "val_sharpness", "Validation sharpness proxy")
plot_metric(history_df, "mean_grad_norm", "Mean gradient norm")
plot_metric(history_df, "mean_eps_norm", "Mean perturbation norm")

In [ ]:
# Accuracy against wall-clock time gives a hardware-specific but practical view.
plt.figure(figsize=(8, 5))
for method in METHODS:
    sub = history_df[history_df["method"] == method]
    grouped = sub.groupby("epoch")[["wall_time_sec", "test_accuracy"]].mean()
    plt.plot(grouped["wall_time_sec"] / 60, 100 * grouped["test_accuracy"], label=method)
plt.xlabel("Mean wall time (minutes)")
plt.ylabel("Test accuracy (%)")
plt.title("Accuracy vs wall time")
plt.legend()
plt.tight_layout()
plt.show()

## Interpretation checklist

Before drawing a conclusion, check all of the following:

1. **Validation selection:** compare test performance at the validation-selected epoch, not at the best test epoch.
2. **Compute:** report both epoch-matched and training-gradient-matched results. SAM has twice the backward passes per update.
3. **Stability:** inspect individual seeds, not only means. Expand beyond three seeds if differences are small.
4. **Tuning:** verify that none of the selected parameters sits at the edge of its grid. If it does, widen the grid.
5. **Branch probability:** report `gda_descent_prob` with every GDA result. At `1`, only descent/no-op pairs occur; at `0`, only ascent/descent pairs occur. Changing this value departs from the paper's equal branch mixture.
6. **GDA dynamics:** inspect `mean_alpha`, `mean_beta`, and `mean_eps_norm`. Persistent \(y\) sitting at \(\rho\) is expected but informative.
7. **Sharpness:** treat parameter-space sharpness as a diagnostic. It depends on the chosen radius, batch, parameterization, and BatchNorm treatment.
8. **Protocol:** the canonical protocol is faithful to standard SAM practice and to the supplied GDA implementation, but GDA lacks momentum/weight decay by definition here. Run the isolated protocol if you need an update-rule-only ablation.
9. **Scope:** one dataset/network pair is evidence, not a universal ranking. CIFAR-100 or a WideResNet is a natural follow-up.